
</br>
<font size="12">Estimating admixture proportions using human 1KGP data</font>

<img src="https://upload.wikimedia.org/wikipedia/commons/e/ec/1000_Genomes_Project.svg" alt="1000 Genomes Project population sampling map" style="max-width: 850px; width: 100%; margin: 1em 0;">

In this exercise we will estimate ancestry proportions from a subset of human 1000 Genomes data. The data set has already been converted to PLINK format and subset to 12 individuals from each of 16 populations. Several populations are expected to be admixed.

The main aim is not just to run ADMIXTURE, but also to see why it is important to check convergence across multiple random seeds.


In [ ]:
# ------------------------------------------------------------
# ENVIRONMENT SETUP
# EDIT THESE PATHS WHEN MOVING THIS NOTEBOOK TO ANOTHER SYSTEM
# ------------------------------------------------------------

# Folder containing the admixture_notebook_files directory.
# If you move the exercise, set this to the parent directory of admixture_notebook_files.
COURSE_DIR=/course/chinacourse2026/shared/data

# Folder containing the small input bundle and precomputed results
NOTEBOOK_DATA_DIR=${COURSE_DIR}/admixture_notebook_files
INPUT_DATA_DIR=${NOTEBOOK_DATA_DIR}/input
PRECOMP_RESULTS_DIR=${NOTEBOOK_DATA_DIR}/precomputed

# Folder where this notebook will copy files and run analyses
WORK_DIR=~/sysu_day4_admixture

# Number of threads students should use when running commands interactively
THREADS=4

# Main model settings used in this notebook
K=4
BAD_SEED=2
BEST_SEED=5

# Local ancestry settings used in the final hapla/fatash section
LAI_PREFIX=chr1_noEAS_w256_l0005
LAI_K=3
LAI_SEED=5
HAPLA_WINDOW_SIZE=256
HAPLA_LAMBDA=0.005

# Save the configuration so Bash, R, and Python cells can all read the same paths.
CONFIG_FILE=~/.sysu_admixture_paths
cat > ${CONFIG_FILE} <<EOF
COURSE_DIR=${COURSE_DIR}
NOTEBOOK_DATA_DIR=${NOTEBOOK_DATA_DIR}
INPUT_DATA_DIR=${INPUT_DATA_DIR}
PRECOMP_RESULTS_DIR=${PRECOMP_RESULTS_DIR}
WORK_DIR=${WORK_DIR}
THREADS=${THREADS}
K=${K}
BAD_SEED=${BAD_SEED}
BEST_SEED=${BEST_SEED}
LAI_PREFIX=${LAI_PREFIX}
LAI_K=${LAI_K}
LAI_SEED=${LAI_SEED}
HAPLA_WINDOW_SIZE=${HAPLA_WINDOW_SIZE}
HAPLA_LAMBDA=${HAPLA_LAMBDA}
EOF

cat ${CONFIG_FILE}


# Software and data

We will use PLINK, PCAone, ADMIXTURE, evalAdmix, and R plotting functions. First check that the programs are available.


In [ ]:
echo --programs that are installed:--
which admixture
which plink
which PCAone
which evalAdmix
which hapla



## Data sets

Make a working directory in your home folder and copy the starter files for the exercise. A lot of results throughout the exercise have been precomputed due to time constraints, but the commands used to generate them are shown.


In [ ]:
CONFIG_FILE=~/.sysu_admixture_paths

if [ ! -f "${CONFIG_FILE}" ]; then
  echo "Could not find ${CONFIG_FILE}. Run the first configuration cell before this one."
  false
else
  source "${CONFIG_FILE}"

  mkdir -p "${WORK_DIR}"
  cd "${WORK_DIR}"

  if [ ! -d "${INPUT_DATA_DIR}" ]; then
    echo "Could not find INPUT_DATA_DIR=${INPUT_DATA_DIR}"
    echo "Edit COURSE_DIR in the first notebook cell so it points to the parent folder of admixture_notebook_files."
    echo "Current COURSE_DIR=${COURSE_DIR}"
    false
  else
    cp -a "${INPUT_DATA_DIR}/." .

    echo -- starter files copied --
    find "${INPUT_DATA_DIR}" -maxdepth 1 -type f | sed "s#^${INPUT_DATA_DIR}/##" | sort
  fi
fi


In [ ]:
# set up R working space from the path configuration written in the first cell
config <- readLines(path.expand("~/.sysu_admixture_paths"))
get_config <- function(key) sub(paste0("^", key, "="), "", config[grepl(paste0("^", key, "="), config)][1])
work_d <- path.expand(get_config("WORK_DIR"))
setwd(work_d)


In [ ]:
# set up python working space from the path configuration written in the first cell
import os
from pathlib import Path

config = {}
for line in Path("~/.sysu_admixture_paths").expanduser().read_text().splitlines():
    key, value = line.split("=", 1)
    config[key] = value

work_d = os.path.expanduser(config["WORK_DIR"])
os.chdir(work_d)



## Metadata and population labels

The PLINK `.fam` file describes the individuals in the genotype data. The labels file maps individuals to population and broad ancestry region.


In [ ]:
echo -- number of individuals in fam file --
wc -l human_autosomes_12pp_pcaoneLD02.fam

echo -e "
-- first 10 lines of fam file --"
head human_autosomes_12pp_pcaoneLD02.fam

echo -e "
-- first 10 lines of label file: sample population region --"
head human_autosomes_12pp_pcaoneLD02.labels.tsv

echo -e "
-- population counts --"
awk '{print $2}' human_autosomes_12pp_pcaoneLD02.labels.tsv | sort | uniq -c

echo -e "
-- broad region counts --"
awk '{print $3}' human_autosomes_12pp_pcaoneLD02.labels.tsv | sort | uniq -c



## The BIM file

The `.bim` file describes the genetic variants. This data set has already been pruned for LD with PCAone, so the variant count is much smaller than the original genome-wide BCF data.


In [ ]:
echo -- number of variants in PCAone-pruned data --
wc -l human_autosomes_12pp_pcaoneLD02.bim

echo -e "
-- first 10 variants --"
echo -e "CHR	variantID	CM	Position	allele_1	allele_2"
head human_autosomes_12pp_pcaoneLD02.bim

echo -e "
-- variants per chromosome --"
awk '{n[$1]++} END {for (c in n) print c, n[c]}' human_autosomes_12pp_pcaoneLD02.bim | sort -k1,1n



Run the code below to start a short quiz about the input data.


In [ ]:
from jupyterquiz import display_quiz
display_quiz('admixture_quiz1.json')



## LD pruning with PCAone

ADMIXTURE assumes that markers are approximately independent. It is therefore common to prune variants in linkage disequilibrium before running ADMIXTURE.

Here we use PCAone for LD pruning because it estimates LD after accounting for population structure. This matters for data sets with individuals from multiple populations: ordinary LD pruning can confuse population structure with linkage disequilibrium.

The `-k` value in PCAone is the number of principal components used to model population structure before LD is estimated from residuals. It is not the same as ADMIXTURE's K, and it should not be interpreted as an expected number of ancestry components. Here we use `-k 10` as a conservative choice to capture several broad and within-region structure axes in the 1KGP subset. In practice you might want to be careful with this number, but plotting a PCA of your data, which you will learn about in the afternoon session, can help with this.

The current PCAone version uses a two-step workflow: first compute ancestry-adjusted residuals, then prune variants using those residuals. The commands are shown below, but the results are precomputed for this exercise.


In [ ]:
# These commands were used to generate the PCAone-pruned data.
# They are shown for reference and are not run in the notebook.
# In this command, -k is the number of PCs used for ancestry-adjusted LD, not ADMIXTURE K.

# PCAone -b human_autosomes_12pp_ids \
#   -k 10 \
#   -D \
#   --ld-stats 0 \
#   -n ${THREADS} \
#   -o pcaone_autosomes_k10

# PCAone -B pcaone_autosomes_k10.residuals \
#   --match-bim pcaone_autosomes_k10.mbim \
#   --ld-r2 0.2 \
#   --ld-bp 1000000 \
#   -n ${THREADS} \
#   -o pcaone_autosomes_k10_prune_ld02

# Use the precomputed PCAone pruning result for the rest of the exercise.
cp -a "${PRECOMP_RESULTS_DIR}/pcaone/." .

echo -- number of variants kept by PCAone pruning --
wc -l pcaone_autosomes_k10_prune_ld02.ld.prune.in

echo -e "
-- number of variants in extracted PLINK data --"
wc -l human_autosomes_12pp_pcaoneLD02.bim


### Quick check: LD pruning and ADMIXTURE setup


In [ ]:
from jupyterquiz import display_quiz
display_quiz('admixture_quiz2_ld_admixture.json')



## ADMIXTURE

ADMIXTURE estimates each individual's ancestry proportions under a model with K ancestral populations. For this exercise we start with K=4 because we are working with data contains four broad regions: Africa (AFR), America (AMR), East Asia (EAS), and Europe (EUR).

The populations labelled AMR are admixed populations from the Americas. Therefore, we should not expect a simple one-to-one interpretation where each region becomes one pure component.

ADMIXTURE will return a model for the K you ask it to fit, even when that K is not a good description of the data. A clean-looking barplot is therefore not enough by itself: the solution may be unstable, underfit, or biologically misleading.


In [ ]:
admixture --help | head -40



ADMIXTURE starts from a random initial guess. Some seeds converge to a good solution and some can get stuck in a local optimum. We will first inspect a deliberately bad seed.

<img src="https://i.sstatic.net/GPErf.gif" alt="Animation of an optimizer moving across a bumpy likelihood landscape" width="520">

The above picture is only a metaphor, most likelihood landscapes will have many more dimensions, but the idea is useful: different random seeds can start the optimization in different parts of a complicated likelihood landscape, so they may end at different local optima. This is why we compare log likelihoods across many seeds instead of trusting a single ADMIXTURE run.


In [ ]:
# Command used to generate the bad seed result shown below.
# This is shown for reference and is not run in the notebook.

# admixture --seed ${BAD_SEED} -j ${THREADS} human_autosomes_12pp_pcaoneLD02.bed ${K}

# Load the precomputed bad-seed ADMIXTURE result.
cp -a "${PRECOMP_RESULTS_DIR}/bad_seed/." .

cat human_autosomes_12pp_pcaoneLD02.4.bad_seed2.log | tail -20



### Plotting a "random" seed

The plot below shows ancestry proportions from seed 2. The populations are ordered roughly geographically: continental African populations, African diaspora populations, European populations, East Asian populations, and admixed American populations.


In [ ]:
library("repr")
options(repr.plot.width=17, repr.plot.height=5)
source("./visFuns.R")

labels <- read.table("human_autosomes_12pp_pcaoneLD02.labels.tsv", stringsAsFactors=FALSE)
pop <- labels[,2]
pop_order <- c("GWD", "MSL", "YRI", "ESN", "ACB", "ASW",
               "GBR", "CEU", "IBS", "TSI",
               "CHB", "CHS", "JPT",
               "MXL", "CLM", "PEL")
ord <- unlist(lapply(pop_order, function(p) which(pop == p)))

admix_cols <- c(AMR="#FF7F00", AFR="#4DAF4A", EUR="#377EB8", EAS="#984EA3")

reorder_components <- function(q) {
  ref <- c("PEL", "YRI", "GBR", "CHB")
  kord <- sapply(ref, function(p) which.max(colMeans(q[pop == p,,drop=FALSE])))
  if(length(unique(kord)) == ncol(q)) q[,kord] else q
}

plot_geo_admix <- function(q, title) {
  plotAdmix(q, pop=pop, ord=ord, rotatelab=45, padj=0.12,
            cex.lab=1.0, cex.main=1.3, main=title,
            colorpal=unname(admix_cols[c("AMR", "AFR", "EUR", "EAS")]))
}

q_bad <- reorder_components(as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.4.bad_seed2.Q")))
plot_geo_admix(q_bad, "ADMIXTURE proportions, K = 4, bad seed = 2")




- Does this result look plausible?
- Which populations or regions look strange?
- What could make ADMIXTURE produce a result like this?


## evalAdmix

evalAdmix diagnoses how well an ADMIXTURE model predicts the genotypes. It looks for correlations in residuals after fitting the model. Strong residual correlations tell us that individuals or populations are more alike (positive values) or different (negative values) than expected under the model. This in turn can indicate that the model is missing structure or that the ADMIXTURE run found a poor solution.

Run evalAdmix for the seed we just looked at. With four threads this should take a couple of minutes on the prepared data set, so it is normal if the cell sits for a little while before printing the final lines.


In [ ]:
# Run evalAdmix for the bad seed. If needed, the precomputed fallback is:
# cp -a "${PRECOMP_RESULTS_DIR}/evaladmix/human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval" .

/usr/bin/time -p evalAdmix \
  -plink human_autosomes_12pp_pcaoneLD02 \
  -fname human_autosomes_12pp_pcaoneLD02.4.bad_seed2.P \
  -qname human_autosomes_12pp_pcaoneLD02.4.bad_seed2.Q \
  -o human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval \
  -P ${THREADS} \
  > human_autosomes_12pp_pcaoneLD02.4.bad_seed2.evalAdmix.log 2>&1

tail -15 human_autosomes_12pp_pcaoneLD02.4.bad_seed2.evalAdmix.log
ls -lh human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval


In [ ]:
options(repr.plot.width=14, repr.plot.height=11)
r_bad <- as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval"))
plotCorRes(r_bad, pop=pop, ord=ord, max_z=0.05,
           rotatelabpop=20, adjlab=.05,
           title="evalAdmix residual correlations, bad seed = 2",
           cex.main=1.2, cex.lab=.9, cex.legend=.9)



- Which population pairs have strong residual correlations?
- Does evalAdmix agree with your visual impression from the ADMIXTURE plot?



## Checking convergence across seeds

To test whether ADMIXTURE has found a good optimum, we run the same K with multiple random seeds and compare log likelihoods. The runs below were precomputed with seeds 1 through 10.


In [ ]:
# Commands used to generate the multi-seed results.
# They are shown for reference and are not run in the notebook.

# mkdir -p multiRunK4
# for seed in 1 2 3 4 5 6 7 8 9 10
# do
#   admixture --seed $seed -j ${THREADS} human_autosomes_12pp_pcaoneLD02.bed ${K} \
#     > multiRunK4/human_autosomes_12pp_pcaoneLD02.4.log_$seed 2>&1
#   mv human_autosomes_12pp_pcaoneLD02.4.Q multiRunK4/human_autosomes_12pp_pcaoneLD02.4.Q_$seed
#   mv human_autosomes_12pp_pcaoneLD02.4.P multiRunK4/human_autosomes_12pp_pcaoneLD02.4.P_$seed
# done

# Load the precomputed likelihood summary from those runs.
mkdir -p multiRunK4
cp -a "${PRECOMP_RESULTS_DIR}/multiRunK4/." multiRunK4/

ls multiRunK4


In [ ]:
echo -- likelihoods sorted from best to worst --
cat multiRunK4/likelihoods_10seeds.tsv


### Quick check: seed convergence


In [ ]:
from jupyterquiz import display_quiz
display_quiz('admixture_quiz3_convergence.json')



## Plotting the best seed

Seed 5 has the highest likelihood among the 10 runs. We now plot that solution and compare it to the bad seed.


In [ ]:
# Load the precomputed best-seed ADMIXTURE result.
cp -a "${PRECOMP_RESULTS_DIR}/best_seed/." .

cat human_autosomes_12pp_pcaoneLD02.4.best_seed5.log | tail -20


In [ ]:
options(repr.plot.width=17, repr.plot.height=5)
q_best <- reorder_components(as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q")))
plot_geo_admix(q_best, "ADMIXTURE proportions, K = 4, best seed = 5")



- How did the result change compared with seed 2?
- Which populations show clear evidence of admixture?
- Do the admixed American populations all have the same ancestry proportions?


## evalAdmix for the best seed

Finally, run evalAdmix for the best seed and compare the model fit.


In [ ]:
# Run evalAdmix for the best seed. If needed, the precomputed fallback is:
# cp -a "${PRECOMP_RESULTS_DIR}/evaladmix/human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval" .

/usr/bin/time -p evalAdmix \
  -plink human_autosomes_12pp_pcaoneLD02 \
  -fname human_autosomes_12pp_pcaoneLD02.4.best_seed5.P \
  -qname human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q \
  -o human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval \
  -P ${THREADS} \
  > human_autosomes_12pp_pcaoneLD02.4.best_seed5.evalAdmix.log 2>&1

tail -15 human_autosomes_12pp_pcaoneLD02.4.best_seed5.evalAdmix.log
ls -lh human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval


In [ ]:
options(repr.plot.width=14, repr.plot.height=11)
r_best <- as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval"))
plotCorRes(r_best, pop=pop, ord=ord, max_z=0.05,
           rotatelabpop=20, adjlab=.05,
           title="evalAdmix residual correlations, best seed = 5",
           cex.main=1.2, cex.lab=.9, cex.legend=.9)


## What if K is too low?

A bad random seed is one way to get a poor fit, but a model can also fit poorly because K is too small. Here we compare the K=4 result to a K=3 ADMIXTURE run. The K=3 run is precomputed, but the command below shows how it was generated.

ADMIXTURE will still return a model when K is too low; the problem is that the model may not be a good or biologically useful description. The likelihoods and evalAdmix residuals are checks on whether the solution is stable enough to interpret.


In [ ]:
# Command used to generate the K=3 result shown below.
# This is shown for reference and is not run in the notebook.

# admixture --seed ${BEST_SEED} -j ${THREADS} human_autosomes_12pp_pcaoneLD02.bed 3

# Load the precomputed K=3 ADMIXTURE result.
cp -a "${PRECOMP_RESULTS_DIR}/k3_underfit/." .

cat human_autosomes_12pp_pcaoneLD02.3.seed5.log | tail -20


In [ ]:
options(repr.plot.width=17, repr.plot.height=5)
q_k3 <- as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.3.seed5.Q"))
ref_k3 <- c("GBR", "CHB", "YRI")
kord_k3 <- sapply(ref_k3, function(p) which.max(colMeans(q_k3[pop == p,,drop=FALSE])))
if(length(unique(kord_k3)) == ncol(q_k3)) q_k3 <- q_k3[,kord_k3]
plotAdmix(q_k3, pop=pop, ord=ord, rotatelab=45, padj=0.12,
          cex.lab=1.0, cex.main=1.3,
          main="ADMIXTURE proportions, K = 3",
          colorpal=unname(admix_cols[c("EUR", "EAS", "AFR")]))



- Which ancestry patterns are forced together when K=3?
- Does K=3 make biological sense for these populations?
- What kinds of structure would you expect evalAdmix to flag for this model?


In [ ]:
# Optional: uncomment this block if you want to run evalAdmix for the K=3 model yourself.

# /usr/bin/time -p evalAdmix \
#   -plink human_autosomes_12pp_pcaoneLD02 \
#   -fname human_autosomes_12pp_pcaoneLD02.3.seed5.P \
#   -qname human_autosomes_12pp_pcaoneLD02.3.seed5.Q \
#   -o human_autosomes_12pp_pcaoneLD02.3.seed5.eval \
#   -P ${THREADS} \
#   > human_autosomes_12pp_pcaoneLD02.3.seed5.evalAdmix.log 2>&1

# Default: copy the precomputed K=3 evalAdmix result.
cp -a "${PRECOMP_RESULTS_DIR}/evaladmix/human_autosomes_12pp_pcaoneLD02.3.seed5.eval" .

# The log is only used to show what a completed run looks like.
cp -a "${PRECOMP_RESULTS_DIR}/k3_underfit/human_autosomes_12pp_pcaoneLD02.3.seed5.evalAdmix.log" . 2>/dev/null || true

if [ -f human_autosomes_12pp_pcaoneLD02.3.seed5.evalAdmix.log ]; then
  tail -15 human_autosomes_12pp_pcaoneLD02.3.seed5.evalAdmix.log
fi
ls -lh human_autosomes_12pp_pcaoneLD02.3.seed5.eval


In [ ]:
options(repr.plot.width=14, repr.plot.height=11)
r_k3 <- as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.3.seed5.eval"))
plotCorRes(r_k3, pop=pop, ord=ord, max_z=0.05,
           rotatelabpop=20, adjlab=.05,
           title="evalAdmix residual correlations, K = 3",
           cex.main=1.2, cex.lab=.9, cex.legend=.9)


- How does the K=3 residual plot differ from the K=4 best-seed residual plot?


In [ ]:
summarize_eval <- function(file) {
  m <- as.matrix(read.table(file))
  m[upper.tri(m, diag=TRUE)] <- NA
  c(mean_abs=mean(abs(m), na.rm=TRUE),
    p95_abs=unname(quantile(abs(m), .95, na.rm=TRUE)),
    max_abs=max(abs(m), na.rm=TRUE))
}

rbind(
  k3_seed5=summarize_eval("human_autosomes_12pp_pcaoneLD02.3.seed5.eval"),
  bad_seed2=summarize_eval("human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval"),
  best_seed5=summarize_eval("human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval")
)


### Quick check: evalAdmix model fit


In [ ]:
from jupyterquiz import display_quiz
display_quiz('admixture_quiz4_evaladmix.json')


## Comparing several K values

K=4 is still the main result in this exercise because it has a simple interpretation for these populations and we checked convergence across seeds. It is still useful to see what happens when K is too low or when K is increased beyond the main model.

The K=2 and K=3 examples show underfitting: several ancestry patterns are forced together. The K=5, K=6, and K=7 examples below are exploratory precomputed runs stopped after 15 main iterations to keep this comparison practical. They are not meant as final, converged model choices. They show a different danger: as K increases, ADMIXTURE can create increasingly fine-scale components that are tempting to label, even when they may reflect sampling, local optima, or structure that is not relevant to the question being asked.


In [ ]:
source ~/.sysu_admixture_paths
cd "${WORK_DIR}"

cp -a "${PRECOMP_RESULTS_DIR}/k_ladder/." .

echo -- K ladder files --
ls -lh \
  human_autosomes_12pp_pcaoneLD02.2.seed5.Q \
  human_autosomes_12pp_pcaoneLD02.3.seed5.Q \
  human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q \
  human_autosomes_12pp_pcaoneLD02.5.seed5_iter15.Q \
  human_autosomes_12pp_pcaoneLD02.6.seed5_iter15.Q \
  human_autosomes_12pp_pcaoneLD02.7.seed5_iter15.Q

echo
echo -- K ladder run notes --
cat k_ladder_summary.tsv


In [ ]:
# all of this is just plotting code to make the figure look a little nicer,
# dont worry about it if you dont understand it

options(repr.plot.width=15, repr.plot.height=11)
source("./visFuns.R")

labels_k <- read.table("human_autosomes_12pp_pcaoneLD02.labels.tsv",
                       stringsAsFactors=FALSE)
pop_k <- labels_k[, 2]
pop_order_k <- c("GWD", "MSL", "YRI", "ESN", "ACB", "ASW",
                 "GBR", "CEU", "IBS", "TSI",
                 "CHB", "CHS", "JPT",
                 "MXL", "CLM", "PEL")
ord_k <- orderInds(pop=pop_k, popord=pop_order_k)

q_files_k <- c(
  "2"="human_autosomes_12pp_pcaoneLD02.2.seed5.Q",
  "3"="human_autosomes_12pp_pcaoneLD02.3.seed5.Q",
  "4"="human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q",
  "5"="human_autosomes_12pp_pcaoneLD02.5.seed5_iter15.Q",
  "6"="human_autosomes_12pp_pcaoneLD02.6.seed5_iter15.Q",
  "7"="human_autosomes_12pp_pcaoneLD02.7.seed5_iter15.Q"
)

admix_cols_k <- c(AMR="#FF7F00", AFR="#4DAF4A", EUR="#377EB8", EAS="#984EA3")
extra_cols_k <- c("#A65628", "#F781BF", "#999999", "#E41A1C")

pick_components_k <- function(q, anchors, anchor_cols) {
  used <- integer(0)
  cols <- character(0)
  for (i in seq_along(anchors)) {
    anchor <- anchors[i]
    means <- colMeans(q[pop_k == anchor, , drop=FALSE])
    means[used] <- -Inf
    k <- which.max(means)
    if (is.finite(means[k]) && !(k %in% used)) {
      used <- c(used, k)
      cols <- c(cols, anchor_cols[i])
    }
    if (length(used) == ncol(q)) break
  }
  extra <- setdiff(seq_len(ncol(q)), used)
  list(order=c(used, extra), colors=c(cols, extra_cols_k[seq_along(extra)]))
}

component_setup_k <- function(q, k_label) {
  if (k_label == "2") {
    return(pick_components_k(q, anchors=c("YRI"),
                             anchor_cols=unname(admix_cols_k["AFR"])))
  }
  if (k_label == "3") {
    return(pick_components_k(q, anchors=c("YRI", "GBR", "CHB"),
                             anchor_cols=unname(admix_cols_k[c("AFR", "EUR", "EAS")])))
  }
  pick_components_k(q, anchors=c("PEL", "YRI", "GBR", "CHB", "ACB", "MXL", "IBS", "CLM"),
                    anchor_cols=c(unname(admix_cols_k[c("AMR", "AFR", "EUR", "EAS")]), extra_cols_k))
}

layout(matrix(seq_along(q_files_k), ncol=1))
for (i in seq_along(q_files_k)) {
  k_label <- names(q_files_k)[i]
  q <- as.matrix(read.table(q_files_k[i]))
  setup_k <- component_setup_k(q, k_label)
  q <- q[, setup_k$order, drop=FALSE]

  par(mar=c(ifelse(i == length(q_files_k), 5.5, 1.1), 4.6, 2.0, 1),
      xpd=FALSE)
  h <- barplot(t(q)[, ord_k], col=setup_k$colors,
               space=0, border=NA, axes=FALSE,
               ylab=ifelse(i == ceiling(length(q_files_k) / 2),
                           "Admixture proportions", ""),
               main=paste0("K = ", k_label),
               cex.main=1.1, cex.lab=1.0)
  axis(2, at=c(0, 0.5, 1), las=1, cex.axis=0.75)
  abline(v=cumsum(as.numeric(table(factor(pop_k[ord_k], levels=unique(pop_k[ord_k]))))),
         col="black", lwd=0.7)

  if (i == length(q_files_k)) {
    pop_centers <- tapply(h, factor(pop_k[ord_k], levels=unique(pop_k[ord_k])), mean)
    text(pop_centers, -0.08, names(pop_centers), srt=45, adj=1,
         cex=0.7, xpd=NA)
  }
}
layout(1)



- Which population structure is merged when K is too low?
- Which extra components at K=5 to K=7 look like plausible fine-scale structure, and which look harder to defend?
- Why does the likelihood generally improve as K increases, and why is that not enough to choose the largest K?
- What additional checks would you want before interpreting one of the higher-K models?


## A short look at local ancestry with hapla/fatash

Genome-wide ADMIXTURE summarizes each individual with one set of proportions. Local ancestry asks a different question: along a chromosome, which component is most likely for each haplotype segment?

Here we use the methods hapla and fatash on chromosome 1. The first and more computationally expensive haplotype clustering step has already been run, because it takes a little too long for an exercise on this data set. The hapla article in the reading list goes into more detail, but this step can be seen as preprocessing step where haplotypes(each individual has two) from phased genotype data is grouped by similarity in windows along each chromosome. 

For this short example we use the data set from before, but without the east Asian populations: African, European, and admixed American populations, excluding JPT, CHS, and CHB. The American-related component is therefore a proxy learned from admixed American populations, not an unadmixed Native American reference panel.
Unlike most other local ancestry methods, hapla does not split individuals into labeled reference and query sets. All individuals are clustered and modeled together.


In [ ]:
source ~/.sysu_admixture_paths
cd "${WORK_DIR}"

# Copy precomputed hapla cluster files and fallback admix/fatash results.
cp -a "${PRECOMP_RESULTS_DIR}/lai/." .

echo -- hapla cluster command used to prepare these files --
echo "hapla cluster \\
  --bcf chr1_noEAS.bcf \\
  --size ${HAPLA_WINDOW_SIZE} \\
  --lmbda ${HAPLA_LAMBDA} \\
  --threads ${THREADS} \\
  --out ${LAI_PREFIX}"

echo
echo -- precomputed hapla cluster files --
ls -lh ${LAI_PREFIX}.bca ${LAI_PREFIX}.win ${LAI_PREFIX}.ids

echo
echo -- first windows --
head ${LAI_PREFIX}.win

echo
echo -- number of windows --
tail -n +2 ${LAI_PREFIX}.win | wc -l


- From looking at the `.win` file, what do you suppose that each row represents?
- Why might retaining local haplotype information be useful for local ancestry inference?


Next, run ADMIXTURE-like inference on the hapla clusters. This works similarly to ADMIXTURE on snps, but instead of each site having two possible alleles, each window might have 50+ different clusters/groups. This estimates a global Q matrix and a P matrix for the clustered haplotypes. It is fast here and should take around 10 seconds.

This hapla example is still based only on chromosome 1, because the clustering step above used `chr1_noEAS.bcf`. The Q matrix is therefore a chromosome-1 haplotype-cluster summary, not a genome-wide estimate like the earlier SNP-based ADMIXTURE run.


In [ ]:
source ~/.sysu_admixture_paths
cd "${WORK_DIR}"

/usr/bin/time -p hapla admix \
  --clusters ${LAI_PREFIX} \
  --K ${LAI_K} \
  --seed ${LAI_SEED} \
  --threads ${THREADS} \
  --out ${LAI_PREFIX} \
  > ${LAI_PREFIX}.admix.log 2>&1

tail -15 ${LAI_PREFIX}.admix.log
ls -lh ${LAI_PREFIX}.K${LAI_K}.s${LAI_SEED}.Q ${LAI_PREFIX}.K${LAI_K}.s${LAI_SEED}.P


In [ ]:
#

options(repr.plot.width=17, repr.plot.height=6)

lai_prefix <- get_config("LAI_PREFIX")
lai_k <- get_config("LAI_K")
lai_seed <- get_config("LAI_SEED")

labels_lai <- read.table("chr1_noEAS.labels.tsv", header=FALSE,
                         col.names=c("sample", "population", "study_region"),
                         stringsAsFactors=FALSE)
ids_lai <- read.table(paste0(lai_prefix, ".ids"), header=FALSE,
                      col.names="sample", stringsAsFactors=FALSE)
stopifnot(identical(ids_lai$sample, labels_lai$sample))

q_hapla_raw <- as.matrix(read.table(paste0(lai_prefix, ".K", lai_k, ".s", lai_seed, ".Q")))
pop_lai <- labels_lai$population
pop_order_lai <- c("GWD", "MSL", "YRI", "ESN", "ACB", "ASW",
                   "GBR", "CEU", "IBS", "TSI",
                   "MXL", "CLM", "PEL")
component_cols_lai <- c("#4DAF4A", "#377EB8", "#FF7F00")
component_names_lai <- c("African-related", "European-related", "American-related proxy")
component_ids_lai <- c("AFR_like", "EUR_like", "AMR_proxy")

choose_component <- function(anchor_pops, used) {
  means <- colMeans(q_hapla_raw[pop_lai %in% anchor_pops, , drop=FALSE])
  means[used] <- -Inf
  which.max(means)
}

k_afr <- choose_component(c("YRI", "ESN", "GWD", "MSL"), integer(0))
k_eur <- choose_component(c("GBR", "CEU", "IBS", "TSI"), k_afr)
k_amr <- choose_component(c("PEL", "MXL", "CLM"), c(k_afr, k_eur))
kord_lai <- c(k_afr, k_eur, k_amr)
q_hapla <- q_hapla_raw[, kord_lai, drop=FALSE]
colnames(q_hapla) <- component_ids_lai

layout(matrix(c(1, 2), nrow=2), heights=c(1, 5))
par(mar=c(0, 4.8, 0, 1), xpd=NA)
plot.new()
legend("center", fill=component_cols_lai, legend=component_names_lai,
       bty="n", cex=0.9, horiz=TRUE)

par(mar=c(6.5, 4.8, 3.5, 1), xpd=FALSE)
ord_lai <- orderInds(q=q_hapla, pop=pop_lai, popord=pop_order_lai)
h <- barplot(t(q_hapla)[, ord_lai], col=component_cols_lai, space=0, border=NA,
             ylab="Admixture proportions", xlab="",
             main="hapla ADMIXTURE proportions, chr1, K = 3",
             cex.main=1.2, cex.lab=1.2, cex.axis=0.9)
ordered_pop_lai <- pop_lai[ord_lai]
pop_factor_lai <- factor(ordered_pop_lai, levels=unique(ordered_pop_lai))
abline(v=cumsum(as.numeric(table(pop_factor_lai))), col="black", lwd=1.1)
pop_centers_lai <- tapply(h, pop_factor_lai, mean)
text(pop_centers_lai, -0.08, names(pop_centers_lai), srt=45, adj=1,
     cex=0.85, xpd=NA)
layout(1)



- How does this Q matrix compare with the genome-wide ADMIXTURE result from earlier?

Now run fatash. It uses the hapla clusters together with the Q and P matrices (the ouput of the admix step) to infer a local ancestry path for each haplotype. 

For the people who know about HMMs:

Fatash is a Hidden Markov Model where the hidden state that we try to infer is the ancestry in each window, where the transition probabilities are parameterized using the admixture proportions and where the emission probabilities are parameterized by the frequencies of the different haplotype groups/clusters (the P matrix, also from admix). Since we get the parameters from the admix results we dont need a training step for the model and decode it directly to get the path, making this method very fast. (If you dont know what anything in this last paragraph means, dont worry about it)


In [ ]:
source ~/.sysu_admixture_paths
cd "${WORK_DIR}"

/usr/bin/time -p hapla fatash \
  --clusters ${LAI_PREFIX} \
  --qfile ${LAI_PREFIX}.K${LAI_K}.s${LAI_SEED}.Q \
  --pfile ${LAI_PREFIX}.K${LAI_K}.s${LAI_SEED}.P \
  --threads ${THREADS} \
  --out ${LAI_PREFIX} \
  > ${LAI_PREFIX}.fatash.log 2>&1

tail -15 ${LAI_PREFIX}.fatash.log
ls -lh ${LAI_PREFIX}.path


The plot below shows five MXL and five ASW examples. Within each population, individuals are chosen to span the main ancestry-proportion gradient in the hapla Q matrix. The two haplotypes from the same individual are plotted closer together.


In [ ]:
options(repr.plot.width=15, repr.plot.height=11)

win_lai <- read.table(paste0(lai_prefix, ".win"), header=TRUE,
                      comment.char="", stringsAsFactors=FALSE)
path_lai_raw <- as.matrix(read.table(paste0(lai_prefix, ".path"), header=FALSE))
raw_to_plot_lai <- setNames(match(seq_len(ncol(q_hapla_raw)), kord_lai),
                            0:(ncol(q_hapla_raw) - 1))
path_lai <- matrix(raw_to_plot_lai[as.character(as.integer(path_lai_raw))],
                   nrow=nrow(path_lai_raw), ncol=ncol(path_lai_raw))

select_examples <- function(target_pop, component, n=5, keep=NULL) {
  idx <- which(pop_lai == target_pop)
  if (!is.null(keep)) {
    idx_keep <- idx[keep(idx)]
    if (length(idx_keep) >= n) idx <- idx_keep
  }
  idx <- idx[order(q_hapla[idx, component])]
  if (length(idx) <= n) return(idx)
  idx[unique(round(seq(1, length(idx), length.out=n)))]
}

example_groups <- list(
  MXL=select_examples("MXL", "AMR_proxy", n=5,
                      keep=function(idx) q_hapla[idx, "AFR_like"] < 0.05),
  ASW=select_examples("ASW", "AFR_like", n=5,
                      keep=function(idx) q_hapla[idx, "AMR_proxy"] < 0.05)
)

hap_gap <- 0.24
ind_gap <- 0.72
pop_gap <- 0.95
y <- 0
plot_rows <- data.frame()
for (group_name in names(example_groups)) {
  if (nrow(plot_rows) > 0) y <- y - pop_gap
  for (idx in example_groups[[group_name]]) {
    plot_rows <- rbind(
      plot_rows,
      data.frame(sample_idx=idx, hap=1, y=y, population=group_name),
      data.frame(sample_idx=idx, hap=2, y=y - hap_gap, population=group_name)
    )
    y <- y - ind_gap
  }
}

plot_haplotype <- function(hap_idx, y, label) {
  states <- path_lai[hap_idx, ]
  runs <- rle(states)
  end_i <- cumsum(runs$lengths)
  start_i <- c(1, head(end_i, -1) + 1)
  segments(win_lai$START[start_i] / 1e6, y, win_lai$END[end_i] / 1e6, y,
           col=component_cols_lai[runs$values], lwd=8, lend=1)
  text(min(win_lai$START) / 1e6 - 5, y, label, xpd=NA, adj=1, cex=0.65)
}

layout(matrix(c(1, 2), nrow=2), heights=c(1, 8))
par(mar=c(0, 9.5, 0, 1), xpd=NA)
plot.new()
legend("center", fill=component_cols_lai, legend=component_names_lai,
       bty="n", cex=0.9, horiz=TRUE)

par(mar=c(4.8, 9.5, 3.2, 1))
plot_x_lai <- c(min(win_lai$START) / 1e6, max(win_lai$END) / 1e6 + 24)
plot(plot_x_lai, range(plot_rows$y) + c(-0.45, 0.45),
     type="n", yaxt="n", ylab="", xlab="Chr1 position (Mb)",
     main="fatash local ancestry paths, chr1 examples",
     bty="n")
abline(h=plot_rows$y, col="grey92", lwd=0.4)

for (idx in unlist(example_groups)) {
  individual_rows <- plot_rows[plot_rows$sample_idx == idx, ]
  q_label <- paste(sprintf("%s %.2f", c("AFR", "EUR", "AMR"), q_hapla[idx, ]),
                   collapse=", ")
  for (hap in 1:2) {
    y_hap <- individual_rows$y[individual_rows$hap == hap]
    plot_haplotype(2 * idx - (2 - hap), y_hap,
                   paste0(labels_lai$sample[idx], " ", pop_lai[idx], " hap", hap))
  }
  text(plot_x_lai[2] - 1, mean(individual_rows$y), q_label,
       adj=1, cex=0.55, col="grey20")
}
layout(1)




### Quick check: local ancestry


In [ ]:
from jupyterquiz import display_quiz
display_quiz('admixture_quiz5_lai.json')


Note: A hapla-specific evalAdmix-style diagnostic is under development, but it is not included in this exercise.
